In [ ]:
!pip install tensorboard datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 12.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 17.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 9.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 8.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 18.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.31.0
    Uninstalling requests-2.31.0:
      Successfully uninstalled requests-2.31.0
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 14.0.2
    Uninstalling pyarrow-14.0.2:
      Successfully uninstalled pyarrow-14.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 24

In [ ]:
import pandas as pd
import nltk
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import tensorflow as tf
from transformers import BertTokenizer, TFBertModel

In [ ]:
# Load your dataset
file_path = "/content/preprocessed_text_reddit.csv"
data = pd.read_csv(file_path)

# Ensure your DataFrame has 'text' and 'label' columns
new_df = pd.DataFrame()
new_df['text'] = data['text']
new_df['label'] = data['label'].values.tolist()

# Download NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

def preprocess_text(text):
    # Lowercasing
    text = text.lower()

    # Removing punctuation
    text = re.sub(r'[^\w\s]', '', text)

    # Tokenization
    tokens = word_tokenize(text)

    # Removing stop words
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]

    # Lemmatization
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    # Join tokens back to a single string
    preprocessed_text = ' '.join(tokens)

    return preprocessed_text

# Apply preprocessing
new_df['text'] = new_df['text'].apply(preprocess_text)
new_df.to_csv("/content/preprocessed_text_reddit.csv")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [ ]:
# Load your dataset
file_path = "/content/preprocessed_text_reddit.csv"
data = pd.read_csv(file_path)

# Ensure your DataFrame has 'text' and 'label' columns
new_df = pd.DataFrame()
new_df['text'] = data['text']
new_df['label'] = data['label'].values.tolist()

# Download NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

def preprocess_text(text):
    # Lowercasing
    text = text.lower()

    # Removing punctuation
    text = re.sub(r'[^\w\s]', '', text)

    # Tokenization
    tokens = word_tokenize(text)

    # Removing stop words
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]

    # Lemmatization
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    # Join tokens back to a single string
    preprocessed_text = ' '.join(tokens)

    return preprocessed_text

# Apply preprocessing
new_df['text'] = new_df['text'].apply(preprocess_text)

# Define a mapping from label integers to their corresponding text labels (if needed)
label_mapping = {
    0: 'label_0',
    1: 'label_1',
    2: 'label_2',
    3: 'label_3'
}
new_df['label_text'] = new_df['label'].map(label_mapping)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [ ]:
# Split the dataset into training (80%), validation (10%), and test (10%) sets with stratification
train_size = 0.8
test_size = 0.1

train_data, temp_data = train_test_split(new_df, train_size=train_size, stratify=new_df['label'], random_state=200)
eval_size = test_size / (1 - train_size)
eval_data, test_data = train_test_split(temp_data, test_size=eval_size, stratify=temp_data['label'], random_state=200)

# Reset the index for all splits
train_data = train_data.reset_index(drop=True)
eval_data = eval_data.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)

# Compute class weights
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(train_data['label']), y=train_data['label'])
class_weights = {i: class_weights[i] for i in range(len(class_weights))}
print("Class weights:", class_weights)

# Convert DataFrames to datasets.Dataset
train_dataset = Dataset.from_pandas(train_data)
eval_dataset = Dataset.from_pandas(eval_data)
test_dataset = Dataset.from_pandas(test_data)

# Create a DatasetDict
dataset_dict = DatasetDict({
    'train': train_dataset,
    'validation': eval_dataset,
    'test': test_dataset
})

# Verify the structure
print(dataset_dict)

# Optionally, save the DatasetDict for later use
dataset_dict.save_to_disk('/content/dataset')

Class weights: {0: 0.34340260995650074, 1: 3.0625, 2: 2.2555555555555555, 3: 3.143805309734513}
DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 2842
    })
    validation: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 355
    })
    test: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 356
    })
})


Saving the dataset (0/1 shards):   0%|          | 0/2842 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/355 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/356 [00:00<?, ? examples/s]

In [ ]:
# Load BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize(batch):
    return tokenizer(batch["text"], padding='max_length', truncation=True, max_length=128)


# Tokenize the dataset
dataset_encoded = dataset_dict.map(tokenize, batched=True)

# Remove the unnecessary columns
dataset_encoded = dataset_encoded.remove_columns(['text', 'label_text'])

# Set format to TensorFlow
dataset_encoded.set_format('numpy', columns=['input_ids', 'attention_mask', 'token_type_ids', 'label'])

def to_tf_dataset(hf_dataset):
    def generator():
        for example in hf_dataset:
            yield (
                {
                    'input_ids': np.array(example['input_ids'], dtype=np.int32),
                    'attention_mask': np.array(example['attention_mask'], dtype=np.int32),
                    'token_type_ids': np.array(example['token_type_ids'], dtype=np.int32)
                },
                np.array(example['label'], dtype=np.int64)
            )
    return tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            {
                'input_ids': tf.TensorSpec(shape=(128,), dtype=tf.int32),
                'attention_mask': tf.TensorSpec(shape=(128,), dtype=tf.int32),
                'token_type_ids': tf.TensorSpec(shape=(128,), dtype=tf.int32)
            },
            tf.TensorSpec(shape=(), dtype=tf.int64)
        )
    )


# Convert splits to TensorFlow datasets
train_tf_dataset = to_tf_dataset(dataset_encoded['train'])
val_tf_dataset = to_tf_dataset(dataset_encoded['validation'])
test_tf_dataset = to_tf_dataset(dataset_encoded['test'])

# Batch and shuffle
BATCH_SIZE = 16
train_tf_dataset = train_tf_dataset.batch(BATCH_SIZE).shuffle(1000).prefetch(tf.data.AUTOTUNE)
val_tf_dataset = val_tf_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_tf_dataset = test_tf_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Verify the structure of a batch
inp, out = next(iter(train_tf_dataset))
print(inp, '\n\n', out)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Map:   0%|          | 0/2842 [00:00<?, ? examples/s]

Map:   0%|          | 0/355 [00:00<?, ? examples/s]

Map:   0%|          | 0/356 [00:00<?, ? examples/s]

{'input_ids': <tf.Tensor: shape=(16, 128), dtype=int32, numpy=
array([[ 101, 3291, 2183, ...,    0,    0,    0],
       [ 101, 3030, 5983, ...,    0,    0,    0],
       [ 101, 7192, 4283, ...,    0,    0,    0],
       ...,
       [ 101, 2144, 2354, ...,    0,    0,    0],
       [ 101, 7592, 4201, ...,    0,    0,    0],
       [ 101, 2583, 4608, ...,    0,    0,    0]], dtype=int32)>, 'attention_mask': <tf.Tensor: shape=(16, 128), dtype=int32, numpy=
array([[1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       ...,
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0]], dtype=int32)>, 'token_type_ids': <tf.Tensor: shape=(16, 128), dtype=int32, numpy=
array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=int32)>} 

 tf.Tensor([1 2 0 0 0 0 3 0 2 2 0

In [ ]:
# Load BERT model
model = TFBertModel.from_pretrained('bert-base-uncased')

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.bias', 'cls.predictions.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

In [ ]:
# Improved model architecture
class ImprovedBERTForClassification(tf.keras.Model):
    def __init__(self, bert_model, num_classes, **kwargs):
        super(ImprovedBERTForClassification, self).__init__(**kwargs)
        self.bert = bert_model
        self.dropout = tf.keras.layers.Dropout(0.5)
        self.dense1 = tf.keras.layers.Dense(256, activation='relu')
        self.norm1 = tf.keras.layers.BatchNormalization()
        self.dense2 = tf.keras.layers.Dense(128, activation='relu')
        self.norm2 = tf.keras.layers.BatchNormalization()
        self.dense3 = tf.keras.layers.Dense(128, activation='relu')
        self.norm3 = tf.keras.layers.BatchNormalization()
        self.fc = tf.keras.layers.Dense(num_classes, activation='softmax')

    def call(self, inputs, training=False):
        bert_outputs = self.bert(inputs, training=training)
        sequence_output = bert_outputs.last_hidden_state  # shape: (batch_size, seq_len, hidden_size)
        pooled_output = tf.reduce_mean(sequence_output, axis=1)  # shape: (batch_size, hidden_size)
        x = self.dropout(pooled_output, training=training)
        x = self.dense1(x)
        x = self.norm1(x, training=training)
        x = self.dense2(x)
        x = self.norm2(x, training=training)
        x = self.dense3(x)
        x = self.norm3(x, training=training)
        return self.fc(x)

    def freeze_bert_layers(self):
        for layer in self.bert.layers:
            layer.trainable = False

    def get_config(self):
        config = super(ImprovedBERTForClassification, self).get_config()
        config.update({
            "bert_model_name": self.bert.name,
            "num_classes": self.fc.units
        })
        return config

    @classmethod
    def from_config(cls, config):
        bert_model = TFBertModel.from_pretrained(config.pop("bert_model_name"))
        return cls(bert_model=bert_model, **config)

In [ ]:
# Initialize the model
num_classes = 4
classifier = ImprovedBERTForClassification(model, num_classes)

# Compile the model
classifier.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

# Callbacks
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10)
model_checkpoint = tf.keras.callbacks.ModelCheckpoint(filepath='best_model', save_best_only=True, save_format='tf')

In [ ]:
# Train the model
history = classifier.fit(
    train_tf_dataset,
    validation_data=val_tf_dataset,
    epochs=10,
    callbacks=[early_stopping, model_checkpoint],
    class_weight=class_weights  # Apply class weights
)

Epoch 1/10
178/178 [==============================] - 147s 651ms/step - loss: 1.4472 - accuracy: 0.3603 - val_loss: 1.2813 - val_accuracy: 0.4225
Epoch 2/10
178/178 [==============================] - 33s 174ms/step - loss: 0.9826 - accuracy: 0.4697 - val_loss: 1.4671 - val_accuracy: 0.4282
Epoch 3/10
178/178 [==============================] - 32s 172ms/step - loss: 0.7841 - accuracy: 0.5299 - val_loss: 1.5715 - val_accuracy: 0.4761
Epoch 4/10
178/178 [==============================] - 32s 173ms/step - loss: 0.6473 - accuracy: 0.5939 - val_loss: 1.6520 - val_accuracy: 0.5127
Epoch 5/10
178/178 [==============================] - 33s 174ms/step - loss: 0.5412 - accuracy: 0.6348 - val_loss: 1.6506 - val_accuracy: 0.5465
Epoch 6/10
178/178 [==============================] - 32s 173ms/step - loss: 0.4725 - accuracy: 0.6499 - val_loss: 1.6804 - val_accuracy: 0.5662
Epoch 7/10
178/178 [==============================] - 33s 173ms/step - loss: 0.4260 - accuracy: 0.6833 - val_loss: 1.6918 - val_a

In [ ]:
# Train the model
history = classifier.fit(
    train_tf_dataset,
    validation_data=val_tf_dataset,
    epochs=10,
    callbacks=[early_stopping, model_checkpoint],
    class_weight=class_weights  # Apply class weights
)

Epoch 1/10
178/178 [==============================] - 32s 171ms/step - loss: 0.3248 - accuracy: 0.7695 - val_loss: 1.7840 - val_accuracy: 0.6310
Epoch 2/10
178/178 [==============================] - 34s 183ms/step - loss: 0.3227 - accuracy: 0.7692 - val_loss: 1.8126 - val_accuracy: 0.6479
Epoch 3/10
178/178 [==============================] - 32s 173ms/step - loss: 0.2900 - accuracy: 0.7977 - val_loss: 1.7867 - val_accuracy: 0.6535
Epoch 4/10
178/178 [==============================] - 32s 171ms/step - loss: 0.2813 - accuracy: 0.8019 - val_loss: 1.8117 - val_accuracy: 0.6592
Epoch 5/10
178/178 [==============================] - 32s 173ms/step - loss: 0.2687 - accuracy: 0.8023 - val_loss: 1.8166 - val_accuracy: 0.6620
Epoch 6/10
178/178 [==============================] - 36s 186ms/step - loss: 0.2538 - accuracy: 0.8195 - val_loss: 1.8116 - val_accuracy: 0.6648
Epoch 7/10
178/178 [==============================] - 33s 172ms/step - loss: 0.2378 - accuracy: 0.8378 - val_loss: 1.8433 - val_ac

In [ ]:
# Train the model
history = classifier.fit(
    train_tf_dataset,
    validation_data=val_tf_dataset,
    epochs=5,
    callbacks=[early_stopping, model_checkpoint],
    class_weight=class_weights  # Apply class weights
)

Epoch 1/5
178/178 [==============================] - 38s 200ms/step - loss: 0.2160 - accuracy: 0.8614 - val_loss: 1.8008 - val_accuracy: 0.6732
Epoch 2/5
178/178 [==============================] - 33s 173ms/step - loss: 0.1960 - accuracy: 0.8733 - val_loss: 1.8016 - val_accuracy: 0.6761
Epoch 3/5
178/178 [==============================] - 32s 171ms/step - loss: 0.2016 - accuracy: 0.8751 - val_loss: 1.8285 - val_accuracy: 0.6789
Epoch 4/5
178/178 [==============================] - 33s 172ms/step - loss: 0.2007 - accuracy: 0.8821 - val_loss: 1.8238 - val_accuracy: 0.6761
Epoch 5/5
178/178 [==============================] - 32s 172ms/step - loss: 0.1954 - accuracy: 0.8797 - val_loss: 1.8490 - val_accuracy: 0.6761


In [ ]:
# # Evaluate the model on the test set
# test_loss, test_accuracy = classifier.evaluate(test_tf_dataset)

# # Print the test accuracy
# print(f'Test Accuracy: {test_accuracy:.4f}')

23/23 [==============================] - 3s 147ms/step - loss: 0.4922 - accuracy: 0.8848
Test Accuracy: 0.8848


In [ ]:
from sklearn.metrics import classification_report
# Get the predictions and true labels for classification report
y_pred = []
y_true = []

for inputs, labels in test_tf_dataset:
    preds = classifier.predict(inputs)
    y_pred.extend(tf.argmax(preds, axis=1).numpy())
    y_true.extend(labels.numpy())

# Print classification report
print(classification_report(y_true, y_pred, target_names=list(label_mapping.values())))


1/1 [==============================] - 0s 62ms/step
              precision    recall  f1-score   support

     label_0       0.92      0.96      0.94       259
     label_1       0.73      0.66      0.69        29
     label_2       0.79      0.68      0.73        40
     label_3       0.81      0.75      0.78        28

    accuracy                           0.88       356
   macro avg       0.81      0.76      0.78       356
weighted avg       0.88      0.88      0.88       356



#Fine Tuning

In [ ]:
fineTune_df=pd.read_csv("/content/combined_dataset.csv")
ft_df = pd.DataFrame()
ft_df['text'] = fineTune_df['transcript']
ft_df['label'] = fineTune_df['label'].values.tolist()

In [ ]:
ft_df['text'][0].lower()

"'is it okay to understand?' 'yes.' in a way, every day on weekdays, i'm working full-time, so i get up in the morning, my kids are small now, so i prepare them, i send them to kindergarten, and then i'm working full-time, and when it's over, i go back to pick them up, and i do all kinds of things around the kids, and then i'm at night, and if i've got a little bit of work left, i'll just stay at home for a little bit more. so in that sense, on saturdays, basically, if you're a kid, and your kid's still a little kid, and you're a kid, and you're a kid, and you're a kid, and you're a kid, and you're a kid, and you're a kid, and you're a kid, and you're a kid, and you're a kid, and you're a kid, and you're a kid, and you're a kid, and you're a kid, and you're a kid, and you're a kid, and you're a kid, and you're a kid, and you're a kid, and you're a kid, and you're a kid, and you're a ki i've been there for pain coffee or something. yes. well, in that sense, since i was born, i haven't h

In [ ]:
def preprocess_text(text):
    if isinstance(text, float):  # Check if the text is a float (which could be NaN)
        text = ""
    else:
        # Lowercasing
        text = text.lower()

        # Removing punctuation
        text = re.sub(r'[^\w\s]', '', text)

        # Tokenization
        tokens = word_tokenize(text)

        # Removing stop words
        stop_words = set(stopwords.words('english'))
        tokens = [word for word in tokens if word not in stop_words]

        # Lemmatization
        lemmatizer = WordNetLemmatizer()
        tokens = [lemmatizer.lemmatize(word) for word in tokens]

        # Join tokens back to a single string
        text = ' '.join(tokens)

    return text
# Apply preprocessing
ft_df['text'] = ft_df['text'].apply(preprocess_text)


In [ ]:
# Define a mapping from label integers to their corresponding text labels
label_mapping = {
    'mild': 0,
    'minimum': 1,
    'moderate': 2,
    'severe': 3
}
ft_df['label']=ft_df['label'].map(label_mapping)

In [ ]:
# Reset the index
ft_data = ft_df.reset_index(drop=True)

# Compute class weights
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(ft_data['label']), y=ft_data['label'])
class_weights = {i: class_weights[i] for i in range(len(class_weights))}
print("Class weights:", class_weights)

# Convert DataFrames to datasets.Dataset
train_dataset = Dataset.from_pandas(ft_data)

# Create a DatasetDict
dataset_dict = DatasetDict({
    'train': train_dataset
})

# Verify the structure
print(dataset_dict)

Class weights: {0: 0.5583333333333333, 1: 0.7282608695652174, 2: 1.8611111111111112, 3: 3.35}
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 67
    })
})


In [ ]:
# Load BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize(batch):
    return tokenizer(batch["text"], padding='max_length', truncation=True, max_length=256)


# Tokenize the dataset
dataset_encoded = dataset_dict.map(tokenize, batched=True)

# Remove the unnecessary columns
dataset_encoded = dataset_encoded.remove_columns(['text',])

# Set format to TensorFlow
dataset_encoded.set_format('numpy', columns=['input_ids', 'attention_mask', 'token_type_ids', 'label'])

def to_tf_dataset(hf_dataset):
    def generator():
        for example in hf_dataset:
            yield (
                {
                    'input_ids': np.array(example['input_ids'], dtype=np.int32),
                    'attention_mask': np.array(example['attention_mask'], dtype=np.int32),
                    'token_type_ids': np.array(example['token_type_ids'], dtype=np.int32)
                },
                np.array(example['label'], dtype=np.int64)
            )
    return tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            {
                'input_ids': tf.TensorSpec(shape=(256,), dtype=tf.int32),
                'attention_mask': tf.TensorSpec(shape=(256,), dtype=tf.int32),
                'token_type_ids': tf.TensorSpec(shape=(256,), dtype=tf.int32)
            },
            tf.TensorSpec(shape=(), dtype=tf.int64)
        )
    )


# Convert to TensorFlow datasets
train_tf_dataset = to_tf_dataset(dataset_encoded['train'])

# Batch and shuffle
BATCH_SIZE = 8
train_tf_dataset = train_tf_dataset.batch(BATCH_SIZE).shuffle(1000).prefetch(tf.data.AUTOTUNE)

# Verify the structure of a batch
inp, out = next(iter(train_tf_dataset))
print(inp, '\n\n', out)

Map:   0%|          | 0/67 [00:00<?, ? examples/s]

{'input_ids': <tf.Tensor: shape=(8, 256), dtype=int32, numpy=
array([[  101, 10047,  3374, ...,  4588,  2409,   102],
       [  101,  2045,  3168, ...,  2215,  2202,   102],
       [  101,  6640,  2692, ...,     0,     0,     0],
       ...,
       [  101,  2008,  2015, ...,  2434,  6139,   102],
       [  101,  2066,  2376, ...,  2282,  2123,   102],
       [  101,  3398,  4921, ...,  2051,  4921,   102]], dtype=int32)>, 'attention_mask': <tf.Tensor: shape=(8, 256), dtype=int32, numpy=
array([[1, 1, 1, ..., 1, 1, 1],
       [1, 1, 1, ..., 1, 1, 1],
       [1, 1, 1, ..., 0, 0, 0],
       ...,
       [1, 1, 1, ..., 1, 1, 1],
       [1, 1, 1, ..., 1, 1, 1],
       [1, 1, 1, ..., 1, 1, 1]], dtype=int32)>, 'token_type_ids': <tf.Tensor: shape=(8, 256), dtype=int32, numpy=
array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=int32)>} 



##Fine Tuning on our dataset

In [ ]:
# Freeze BERT layers
classifier.freeze_bert_layers()

In [ ]:
# Train the model
history = classifier.fit(
    train_tf_dataset,
    validation_data=val_tf_dataset,
    epochs=5,
    callbacks=[early_stopping, model_checkpoint],
    class_weight=class_weights  # Apply class weights
)

Epoch 1/5


9/9 [==============================] - 30s 925ms/step - loss: 2.7834 - accuracy: 0.2090 - val_loss: 1.2846 - val_accuracy: 0.5042
Epoch 2/5
9/9 [==============================] - 7s 839ms/step - loss: 1.5314 - accuracy: 0.3433 - val_loss: 1.0895 - val_accuracy: 0.5831
Epoch 3/5
9/9 [==============================] - 7s 856ms/step - loss: 1.2682 - accuracy: 0.3433 - val_loss: 1.1134 - val_accuracy: 0.5577
Epoch 4/5
9/9 [==============================] - 9s 1s/step - loss: 1.1907 - accuracy: 0.4328 - val_loss: 1.1343 - val_accuracy: 0.5577
Epoch 5/5
9/9 [==============================] - 7s 827ms/step - loss: 1.0335 - accuracy: 0.4478 - val_loss: 1.1540 - val_accuracy: 0.5775


In [ ]:
history = classifier.fit(
    train_tf_dataset,
    validation_data=val_tf_dataset,
    epochs=5,
    callbacks=[early_stopping, model_checkpoint],
    class_weight=class_weights  # Apply class weights
)

Epoch 1/5
9/9 [==============================] - 8s 935ms/step - loss: 0.5259 - accuracy: 0.7612 - val_loss: 1.1858 - val_accuracy: 0.6056
Epoch 2/5
9/9 [==============================] - 8s 882ms/step - loss: 0.3841 - accuracy: 0.7761 - val_loss: 1.2855 - val_accuracy: 0.5944
Epoch 3/5
9/9 [==============================] - 7s 842ms/step - loss: 0.2879 - accuracy: 0.8507 - val_loss: 1.3538 - val_accuracy: 0.5831
Epoch 4/5
9/9 [==============================] - 9s 1s/step - loss: 0.2009 - accuracy: 0.8955 - val_loss: 1.2919 - val_accuracy: 0.6141
Epoch 5/5
9/9 [==============================] - 9s 1s/step - loss: 0.1060 - accuracy: 0.9851 - val_loss: 1.3321 - val_accuracy: 0.6169


In [ ]:
from sklearn.metrics import classification_report
# Get the predictions and true labels for classification report
y_pred = []
y_true = []

for inputs, labels in test_tf_dataset:
    preds = classifier.predict(inputs)
    y_pred.extend(tf.argmax(preds, axis=1).numpy())
    y_true.extend(labels.numpy())

# Print classification report
print(classification_report(y_true, y_pred, target_names=list(label_mapping.keys())))


1/1 [==============================] - 0s 61ms/step
              precision    recall  f1-score   support

        mild       0.93      0.87      0.90       259
     minimum       0.82      0.79      0.81        29
    moderate       0.67      0.70      0.68        40
      severe       0.58      0.93      0.71        28

    accuracy                           0.85       356
   macro avg       0.75      0.82      0.78       356
weighted avg       0.87      0.85      0.85       356



In [ ]:
# Save the trained model
model_path = '/content/my_model.keras'
classifier.save(model_path)
print("Model saved to", model_path)


/usr/local/lib/python3.10/dist-packages/transformers/generation/tf_utils.py:465: UserWarning: `seed_generator` is deprecated and will be removed in a future version.
  warnings.warn("`seed_generator` is deprecated and will be removed in a future version.", UserWarning)


Model saved to /content/my_model.keras


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
model_path = '/content/drive/MyDrive'
classifier.save(model_path)
print("Model saved to", model_path)

Model saved to /content/drive/MyDrive
